# 🕸️ 15 · Bayesian networks I: graphs that factorize probability
### Directed acyclic graphs, conditional probability tables and the factorized joint distribution — why a network of 11 signalling proteins needs 230 numbers instead of 177,146

**BiologyToML** · Part D — Probabilistic graphical models · Module 15 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/15_bayesian_networks_representation.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (graphs are drawn with the toolkit's `draw_graph`; no graph library needed) |
| **Prerequisites** | Modules 02 (joint and conditional probability, the chain rule), 05 (likelihood) and 08 (naïve Bayes) |
| **Estimated time** | 140 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `008_BN_mz_021825` slides 1–37 (+ slide 173, naïve Bayes revisited) — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All probabilities in Sections 1–6 are the numbers printed on the 02-620 slides (alarm network, movie example) or
> small illustrative values that are labelled as such. Section 4 and Section 7 use the real *curated* consensus
> signalling network of Sachs et al. (2005) — its graph only. Section 7.2 simulates data on that graph (labelled
> SIMULATED); the real single-cell measurements are analysed in Modules 16 and 17.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a probabilistic graphical model is (a graph + a probability distribution) and **distinguish** directed models (Bayesian networks) from undirected ones (Markov random fields).
2. **Use** the vocabulary of directed acyclic graphs (parents, children, ancestors, descendants) and **implement** a topological sort that detects cycles.
3. **Write down** the factorized joint distribution $P(X_1,\dots,X_n)=\prod_i P(X_i\mid \mathrm{Pa}(X_i))$ for any DAG and **build** a Bayesian network with the three-step recipe of the slides.
4. **Derive** the chain rule as the *complete* DAG and **count** free parameters (31 vs 10 for the alarm network; 177,146 vs 230 for a signalling network) with a function that works for any DAG.
5. **Compute** joint probabilities by hand and in code — $P(Le{=}1,Li{=}1,S{=}1)=0.12$ and $P(B,\neg E,A,J,\neg M)=0.005355$ — and **check** that the full joint table sums to 1.
6. **Implement** a small reusable Bayesian-network class with exact joint probabilities and ancestral sampling, and **compare** sampled frequencies with the exact distribution.
7. **Connect** Bayesian networks to biology: naïve Bayes, Mendelian inheritance, gene regulatory and signalling networks — and **state** what an edge does and does not mean.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit (alarm CPTs, Sachs network, `draw_graph`) | 5 min |
| 1 · Graph + probability | why joint tables explode; directed vs undirected; smoking; naïve Bayes revisited | 15 min |
| 2 · Directed acyclic graphs | parents, children, descendants; topological sort and cycle detection | 20 min |
| 3 · The joint distribution of a BN | CPTs, the product formula, 0.005355, the full 32-row table | 15 min |
| 4 · Factorising joints | chain rule = complete DAG; 31 vs 10 parameters; scaling up to gene networks | 25 min |
| 5 · Numerical example: movie preference | 0.5 × 0.4 × 0.6 = 0.12; questions answered from the table | 15 min |
| 6 · A reusable Bayesian-network class | `DiscreteBN`: joint, table, ancestral sampling | 20 min |
| 7 · Bayesian networks in biology | Mendel as a BN; signalling networks; what an edge means | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

In 2005, Karen Sachs, Garry Nolan and colleagues measured **11 signalling molecules** — nine phosphorylated proteins
(Raf, Mek, Erk, Akt, PKA, PKC, PLCγ, p38, JNK) and two membrane phospholipids (PIP2, PIP3) — in thousands of
*individual* human T cells by flow cytometry (Sachs *et al.* 2005, *Science*). For brevity we call all eleven
"proteins", as the course toolkit does. Their question was one every cell biologist asks: **which molecules directly
influence which?** The answer they proposed was a *Bayesian network* — a diagram of arrows between the proteins, plus a
probability model attached to it.

Why not simply write down "the probability distribution of 11 proteins"? Suppose every protein is recorded as
*low / medium / high*. A complete table of probabilities for all combinations has $3^{11} = 177{,}147$ cells, i.e.
177,146 numbers to estimate — from a few thousand cells. Hopeless. The arrows rescue us: if each protein is directly
influenced by only a few others, the joint distribution **factorizes** into small local tables, and the number of
parameters drops to a few hundred. The same idea runs through biology:

- **Mendelian inheritance** is a Bayesian network: a child's genotype depends only on the parents' genotypes.
- **Gene regulation**: a target gene's expression depends on the transcription factors that bind its promoter.
- **Naïve Bayes** (Module 08) is the simplest Bayesian network of all.

This notebook builds the representation — graph + local probability tables — from the ground up, exactly following
the first 37 slides of the 02-620 lecture. Module 16 reads independence statements off the graph; Module 17 computes
with it and learns it from the real Sachs data.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They hold the alarm network exactly as on the slides (`ALARM_PARENTS`, `ALARM_CPTS`), the 20-edge consensus signalling network of Sachs et al. 2005, a simulator that generates data on that network, and `draw_graph`, a 30-line Matplotlib function that draws small graphs. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALARM_PARENTS = {"B": (), "E": (), "A": ("B", "E"), "J": ("A",), "M": ("A",)}


ALARM_CPTS = {
    "B": {(): 0.05},
    "E": {(): 0.10},
    "A": {(1, 1): 0.95, (1, 0): 0.85, (0, 1): 0.50, (0, 0): 0.05},
    "J": {(1,): 0.70, (0,): 0.05},
    "M": {(1,): 0.80, (0,): 0.15},
}


SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


def simulate_sachs_like(rng: np.random.Generator, n: int = 2603, kind: str = "continuous") -> pd.DataFrame:
    """Simulated stand-in for the Sachs data: a linear-Gaussian model on the consensus network."""
    order = ["pip3", "plc", "pip2", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]  # topological
    parents = {p: [a for a, b in SACHS_CONSENSUS_EDGES if b == p] for p in SACHS_PROTEINS}
    data = {}
    for node in order:
        value = rng.normal(0, 1, n)
        for par in parents[node]:
            value = value + rng.choice([-1, 1]) * rng.uniform(0.4, 0.8) * data[par]
        data[node] = (value - value.mean()) / value.std()
    df = pd.DataFrame({p: data[p] for p in SACHS_PROTEINS})
    if kind == "discrete":
        df = df.apply(lambda c: pd.qcut(c, 3, labels=False)).astype(int)
    return df


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_sachs_like', 'draw_graph']))

### 📓 Conventions for Modules 15–17

- Every variable in the alarm network is **binary**: 1 = "happened / true", 0 = "did not happen / false". The slides
  write $B$ for "$B=1$" and $\neg B$ ("not B") for "$B=0$", so $P(B,\neg E,A,J,\neg M)$ means
  $P(B{=}1,E{=}0,A{=}1,J{=}1,M{=}0)$.
- `ALARM_PARENTS[node]` is the tuple of the node's parents; `ALARM_CPTS[node][parent values]` is
  $P(\text{node}=1\mid\text{parents})$. Only the probability of "1" is stored, because $P(\text{node}=0\mid\cdot)=1-P(\text{node}=1\mid\cdot)$.
- A graph is a list of **edges** `(parent, child)`; a **parents dictionary** says the same thing node by node.
- `draw_graph(edges, pos)` needs a position $(x, y)\in[0,1]^2$ for every node.

In [ ]:
# ── 0.1 The alarm network as stored in the toolkit ──
print("parents:", ALARM_PARENTS)
for node, table in ALARM_CPTS.items():
    print(f"P({node}=1 | parents) :", table)
ALARM_POS = {"B": (0.2, 0.85), "E": (0.8, 0.85), "A": (0.5, 0.5), "J": (0.25, 0.15), "M": (0.75, 0.15)}

---
## 1 · Graph + probability

> 📍 **Where we are.** Module 02 taught you joint and conditional probabilities; Module 08 used a strong
> independence assumption (naïve Bayes) to classify cancer from genotypes. This section explains the general idea
> behind that trick: a **graph** that says which variables depend directly on which, turning one enormous probability
> table into many small ones.

🧮 **Math refresher — joint, conditional, chain rule.** For random variables $X$ and $Y$ (say, "smoker?" and "lung
cancer?"), the **joint** probability $P(X{=}x, Y{=}y)$ is the probability that both happen. The **conditional**
probability is

$$P(X=x \mid Y=y) = \frac{P(X=x, Y=y)}{P(Y=y)}.$$

*In words:* among the cases where $Y=y$, the fraction in which also $X=x$. Rearranging gives the **product rule**
$P(X,Y)=P(X\mid Y)\,P(Y)$, and applying it repeatedly gives the **chain rule** for $n$ variables:

$$P(X_1,X_2,\dots,X_n) = P(X_1)\,P(X_2\mid X_1)\,P(X_3\mid X_1,X_2)\cdots P(X_n\mid X_1,\dots,X_{n-1}).$$

*In words:* the probability of a whole configuration is the probability of the first variable, times the probability
of the second given the first, and so on — each variable conditioned on **all** the variables before it. (We write
$P(X)$ as shorthand for "the probability of whatever value $X$ takes".) $X$ and $Y$ are **independent** when
$P(X,Y)=P(X)P(Y)$, equivalently $P(X\mid Y)=P(X)$.

How many numbers does a joint distribution need? A table over $n$ binary variables has $2^n$ cells; because the cells
must add up to 1, the last one is determined by the others, so $2^n-1$ numbers are **free**. For variables with
three levels (low/medium/high) it is $3^n-1$.

In [ ]:
# ── 1.1 How big is a full joint probability table? ──
print(f"{'n variables':>12} {'binary: 2^n − 1':>18} {'3 levels: 3^n − 1':>20}")
for n in [1, 2, 3, 5, 11, 20, 30]:
    print(f"{n:>12} {2**n - 1:>18,} {3**n - 1:>20,}")

🔍 **What just happened?** Five binary variables need 31 numbers; 11 three-level proteins need 177,146; 30 binary
genes need over a billion. No experiment can estimate a billion numbers, and no computer can even store the
table for 300 genes. We need **assumptions** that cut the table into pieces — and a language to state them clearly.

That language is the **probabilistic graphical model** (slides 2–4): *probability distribution + graph*.

- **Nodes** are random variables (a gene's expression, a protein's phosphorylation, "burglary happened").
- **Edges** are probabilistic dependencies. Missing edges are the assumptions that save parameters.
- **Directed** graphs (arrows) give **Bayesian networks** (BNs) — the subject of Modules 15–17.
- **Undirected** graphs (plain lines) give **Markov random fields** — Gaussian graphical models (Module 18) are the
  most important example for gene networks.

In [ ]:
# ── 1.2 Two kinds of graphical model (the examples of slide 3) ──
directed_edges = [("A", "C"), ("B", "C"), ("D", "C"), ("A", "D")]
directed_pos = {"A": (0.45, 0.85), "B": (0.1, 0.4), "C": (0.45, 0.35), "D": (0.85, 0.55)}
undirected_edges = [(1, 2), (1, 5), (2, 3), (2, 5), (3, 4), (4, 5), (4, 6)]
undirected_pos = {1: (0.9, 0.7), 2: (0.6, 0.9), 3: (0.25, 0.85), 4: (0.25, 0.45), 5: (0.6, 0.55), 6: (0.05, 0.1)}

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
draw_graph(directed_edges, directed_pos, ax=axes[0],
           title="directed (Bayesian network)\nP(A,B,C,D) = P(A) P(B) P(C|A,B,D) P(D|A)")
draw_graph(undirected_edges, undirected_pos, directed=False, ax=axes[1],
           title="undirected (Markov random field)\n$X_1 \\perp \\{X_3,X_4,X_6\\} \\mid \\{X_2,X_5\\}$")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On the left, every arrow points from a node to a node that depends on it, and the joint
distribution is a product of one factor per node: $A$ and $B$ have no parents, $D$ depends on $A$, and $C$ depends on
its three parents $A, B, D$. On the right there are no arrows; instead, a node is independent of everything else once
its neighbours are known (node 1 has neighbours 2 and 5). The symbol $\perp$ means "is independent of" and the bar
$\mid$ means "given". Module 16 is entirely about reading such statements off a graph.

### 1.1 A causal story: smoking, lung cancer and yellow fingernails (slide 5)

Slide 5 draws one cause with two effects: **S**moking → **Y**ellow finger nails and Smoking → **L**ung cancer, with a
dashed line between Y and L (they are correlated, but neither causes the other). The graph says

$$P(S, Y, L) = P(Y\mid S)\,P(L\mid S)\,P(S) \quad\Longrightarrow\quad Y \perp L \mid S.$$

*In words:* the probability of a configuration is "how likely is this smoking status" times "how likely are these
fingernails for this smoking status" times "how likely is this lung status for this smoking status"; consequently,
once you know whether someone smokes, their fingernails tell you nothing more about their lungs.

⚠️ It is tempting to read this story as a chain S → L → Y ("smoking causes cancer, which causes yellow nails"), but
the slide draws a **common cause** (S → Y and S → L): the nails are stained by the smoke, not by the tumour. Let us
check the claim with numbers. The slides give no CPTs here, so the values
below are **illustrative** (not epidemiological estimates).

In [ ]:
# ── 1.3 Yellow fingernails predict lung cancer — but only through smoking ──
p_S = np.array([0.8, 0.2])                        # P(S=0), P(S=1)            (illustrative numbers)
p_Y_given_S = np.array([[0.95, 0.05],             # row s: P(Y=0|S=s), P(Y=1|S=s)
                        [0.40, 0.60]])
p_L_given_S = np.array([[0.99, 0.01],             # row s: P(L=0|S=s), P(L=1|S=s)
                        [0.85, 0.15]])
joint = p_S[:, None, None] * p_Y_given_S[:, :, None] * p_L_given_S[:, None, :]   # joint[s, y, l]
print(f"the 8 joint probabilities sum to {joint.sum():.6f}")

p_L1 = joint[:, :, 1].sum()
p_L1_Y1 = joint[:, 1, 1].sum() / joint[:, 1, :].sum()
p_L1_Y0 = joint[:, 0, 1].sum() / joint[:, 0, :].sum()
print(f"P(L=1)              = {p_L1:.4f}")
print(f"P(L=1 | Y=1)        = {p_L1_Y1:.4f}   ← yellow nails raise the risk {p_L1_Y1 / p_L1:.1f}-fold")
print(f"P(L=1 | Y=0)        = {p_L1_Y0:.4f}")
for s in (0, 1):
    p_y1 = joint[s, 1, 1] / joint[s, 1, :].sum()
    p_y0 = joint[s, 0, 1] / joint[s, 0, :].sum()
    print(f"P(L=1 | Y=1, S={s})  = {p_y1:.4f}   P(L=1 | Y=0, S={s}) = {p_y0:.4f}   P(L=1 | S={s}) = {p_L_given_S[s, 1]:.4f}")

🔍 **What just happened?** Marginally, yellow nails are a strong *predictor*: $P(L{=}1\mid Y{=}1)=0.115$ versus
$P(L{=}1)=0.038$ overall — a three-fold rise. But within smokers (or within non-smokers) the fingernails change
nothing: $P(L{=}1\mid Y, S{=}1)=0.15$ whatever $Y$ is. The graph predicted this without any arithmetic: the only
route between Y and L passes through S, and knowing S **blocks** it. That is the whole point of drawing the graph —
it tells you which variables "screen off" others, and which correlations are not causal.

✏️ **Your turn.** In cell 1.3, make smoking rare (`p_S = np.array([0.95, 0.05])`) and re-run. Is $P(L{=}1\mid Y{=}1)$
still larger than $P(L{=}1)$? Is $P(L{=}1\mid Y, S{=}1)$ still the same for both values of $Y$? Which numbers depend on
the CPTs, and which statement depends only on the graph?

### 1.2 Naïve Bayes revisited (slide 173)

In Module 08 the naïve Bayes classifier assumed that the genotypes $X_1,\dots,X_J$ are independent given the class
$Y$ (cancer or healthy):

$$P(Y, X_1, \dots, X_J) = P(Y)\prod_{j=1}^{J} P(X_j \mid Y).$$

*In words:* first decide the class, then generate each genotype from its own class-specific table, ignoring the
other genotypes. As a graph, this is $Y$ with an arrow to every $X_j$ and **no** arrows among the $X_j$ — the simplest
Bayesian network. (The slide uses $n$ for the number of features; the course uses $J$.)

🧮 **Math refresher — the product sign.** $\prod_{j=1}^{J} a_j = a_1\times a_2\times\cdots\times a_J$, just as
$\sum_{j=1}^{J} a_j = a_1+a_2+\dots+a_J$.

In [ ]:
# ── 1.4 Naïve Bayes as a graph, and what it saves ──
nb_edges = [("Y", f"X{j}") for j in range(1, 5)]
nb_pos = {"Y": (0.5, 0.85), "X1": (0.1, 0.2), "X2": (0.37, 0.2), "X3": (0.63, 0.2), "X4": (0.9, 0.2)}
fig, ax = plt.subplots(figsize=(4, 2.4))
draw_graph(nb_edges, nb_pos, ax=ax, title="naïve Bayes = the simplest Bayesian network")
plt.tight_layout(); plt.show()

# Y binary, every genotype X_j has 3 levels (0 = AA, 1 = AT, 2 = TT)
print(f"{'J loci':>7} {'full joint 2·3^J − 1':>22} {'naïve Bayes 1 + 2·2·J':>23}")
for J in [3, 10, 20, 30]:
    print(f"{J:>7} {2 * 3**J - 1:>22,} {1 + 2 * 2 * J:>23,}")

🔍 **What just happened?** For the three-locus genotype table of Module 08, the full joint needs $2\cdot 3^3-1=53$
numbers but naïve Bayes needs only 13: one for $P(Y{=}1)$ and, for each locus and each class, two free numbers for the
three genotype probabilities. The saving grows from exponential ($3^J$) to linear ($4J$) — the course's recurring
"$3^n \to 2n$" story. Bayesian networks generalise it: *any* pattern of missing arrows, not just "all features
independent given the class".

Slide 6 shows a real example of the same kind: a network of genes in the **estrogen-receptor pathway** of breast
cancer (ESR1, FOXA1, GATA3, PGR, GREB1, MYC, FOS, …), with the joint distribution written as
$P(\mathrm{NCAPG},\mathrm{FOS},\dots)=P(\mathrm{NCAPG})\,P(\mathrm{FOS}\mid \mathrm{NCAPG})\cdots$ — one factor per
gene, each conditioned on its regulators.

### 📑 Slide connection — `008_BN_mz_021825`, slides 2–8 (and 173)

- **What the slides show.** A probabilistic graphical model = probability distribution + graph. Nodes are random variables, edges are
    probabilistic dependencies; sets of conditional independence assumptions are expressed through the graph
    structure. Directed graphs = Bayesian networks, undirected = Markov random fields (slide 3 shows one of each).
    Examples: smoking → yellow nails and smoking → lung cancer, giving $Y\perp L\mid S$ (slide 5); the
    estrogen-receptor gene network (slide 6); naïve Bayes as the simplest BN (slide 173). Slide 7 lays out the
    lecture: model → inference → learning.
- **What this notebook adds.** Why joint tables explode (printed for 5, 11, 20, 30 variables); the smoking example made numerical with
    illustrative CPTs, showing that a strong correlation disappears once the common cause is known; parameter
    counts for naïve Bayes; a warning not to misread the common cause of slide 5 as a chain.
- **Exam tip.** Be able to write the factorization implied by any small graph (e.g. slide 3: $P(A)P(B)P(D\mid A)
    P(C\mid A,B,D)$) and, conversely, draw the graph from a factorization. The parent set of a node is exactly the
    set of variables to the right of the bar in its factor.

---
## 2 · Directed acyclic graphs

> 📍 **Where we are.** Section 1 introduced graphs informally. A Bayesian network needs a particular kind of graph,
> the **directed acyclic graph** (DAG). This section defines it, gives the vocabulary the rest of Part D uses, and
> writes the algorithm (topological sorting) that both checks a graph for cycles and tells us in which order to
> generate data from a network (Section 6, Module 17).

**The alarm story (slide 9).** You have a burglar alarm at home. It can be triggered by a **B**urglary or, less
reliably, by an **E**arthquake. If the **A**larm goes off, your neighbours **J**ohn and **M**ary may call the
police, and if they call, the **P**olice may come. Each arrow points from a cause to an effect.

A **directed acyclic graph** is a set of nodes and a set of directed edges (arrows) with **no directed cycles**: you can
never follow arrows and come back to where you started (slides 10–11).

| Term | Meaning | In the alarm network |
|---|---|---|
| parents $\mathrm{Pa}(X)$ | nodes with an arrow *into* $X$ | $\mathrm{Pa}(A)=\{B,E\}$ |
| children | nodes $X$ points to | children of $A$: $J, M$ |
| descendants | nodes reachable from $X$ by following arrows | descendants of $A$: $J, M, P$ |
| ancestors | nodes from which $X$ can be reached | ancestors of $P$: $A, B, E, J, M$ |
| root / leaf | node without parents / without children | roots $B, E$; leaf $P$ |

In [ ]:
# ── 2.1 Parents, children, descendants (slide 12) ──
ALARM6_NODES = ["B", "E", "A", "J", "M", "P"]
ALARM6_EDGES = [("B", "A"), ("E", "A"), ("A", "J"), ("A", "M"), ("J", "P"), ("M", "P")]
ALARM6_POS = {"B": (0.2, 0.95), "E": (0.8, 0.95), "A": (0.5, 0.67),
              "J": (0.27, 0.38), "M": (0.73, 0.38), "P": (0.5, 0.08)}


def parents_of(edges, node):
    """Nodes with an arrow into `node`."""
    return sorted({u for u, v in edges if v == node})


def children_of(edges, node):
    """Nodes that `node` points to."""
    return sorted({v for u, v in edges if u == node})


def descendants_of(edges, node):
    """All nodes reachable from `node` by following arrows (depth-first search)."""
    found, stack = set(), [node]
    while stack:
        for child in children_of(edges, stack.pop()):
            if child not in found:
                found.add(child)
                stack.append(child)
    return sorted(found)


def ancestors_of(edges, node):
    """All nodes from which `node` can be reached = descendants in the reversed graph."""
    return descendants_of([(v, u) for u, v in edges], node)


print("parents of A    :", parents_of(ALARM6_EDGES, "A"))
print("children of A   :", children_of(ALARM6_EDGES, "A"))
print("descendants of A:", descendants_of(ALARM6_EDGES, "A"))
print("ancestors of P  :", ancestors_of(ALARM6_EDGES, "P"))

🔍 **What just happened?** The functions reproduce slide 12: the parents of A are B and E, its children J and M, and
its descendants J, M and P. `descendants_of` walks along arrows with a *stack* of nodes still to visit — a
depth-first search, the same idea a biologist uses to trace all downstream targets of a transcription factor.

### 2.1 Why no cycles?

Slide 11 tries two extra arrows. **B → E** is fine ("burglaries cause earthquakes" is silly biology but a legal
graph). **M → E** is not: it closes the loop E → A → M → E. In a Bayesian network every variable is generated from its
parents, so a cycle would be a circular definition — to know E you would need M, which needs A, which needs E.
Mathematically, the product formula of Section 3 would no longer define a valid probability distribution.

⚠️ Real biology is full of **feedback loops** (a kinase activating its own phosphatase, a transcription factor
repressing its own gene). A Bayesian network cannot draw them directly. Two standard escapes appear later in the
course: *unroll the loop over time* (dynamic Bayesian networks and HMMs, Modules 19–20) or *drop the arrows*
(undirected Gaussian graphical models, Module 18).

🧮 **Topological order.** A list of the nodes in which every parent comes before its children (for the alarm network:
B, E, A, J, M, P). A graph has a topological order **if and only if** it has no directed cycle. **Kahn's algorithm**
finds one: repeatedly take out a node that has no remaining incoming arrows, and delete its outgoing arrows. If at
some point every remaining node still has an incoming arrow, those nodes sit on a cycle (or downstream of one).

In [ ]:
# ── 2.2 Topological sorting with cycle detection (Kahn's algorithm) ──
def topological_order(nodes, edges):
    """Return (order, leftover). `order` lists nodes so that parents come before children;
    `leftover` is empty if and only if the graph is acyclic (a DAG)."""
    indegree = {v: 0 for v in nodes}
    for u, v in edges:
        indegree[v] += 1
    ready = [v for v in nodes if indegree[v] == 0]     # nodes with no incoming arrows left
    order = []
    while ready:
        u = ready.pop(0)
        order.append(u)
        for v in children_of(edges, u):                # "delete" u's outgoing arrows
            indegree[v] -= 1
            if indegree[v] == 0:
                ready.append(v)
    return order, [v for v in nodes if v not in order]


def on_a_cycle(nodes, edges):
    """Nodes that can reach themselves by following arrows."""
    return [v for v in nodes if v in descendants_of(edges, v)]


for label, extra in [("alarm network", []), ("+ B→E", [("B", "E")]), ("+ M→E", [("M", "E")])]:
    order, leftover = topological_order(ALARM6_NODES, ALARM6_EDGES + extra)
    print(f"{label:14s} order = {order}   leftover = {leftover}   on a cycle = {on_a_cycle(ALARM6_NODES, ALARM6_EDGES + extra)}")

In [ ]:
# ── 2.3 Slide 11 as pictures ──
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
cases = [("the alarm network (slide 9)", []), ("add B→E: still a DAG ✓", [("B", "E")]),
         ("add M→E: cycle E→A→M→E ✗", [("M", "E")])]
for ax, (title, extra) in zip(axes, cases):
    edges = ALARM6_EDGES + extra
    draw_graph(edges, ALARM6_POS, ax=ax, highlight=on_a_cycle(ALARM6_NODES, edges), title=title)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the alarm network Kahn's algorithm returns the order B, E, A, J, M, P and nothing is
left over. Adding B → E keeps the same order (E already comes after B) and leaves nothing over. Adding M → E stops the
algorithm after B: every remaining node still has an incoming arrow. E, A and M are on the cycle (orange); J and P
are left over only because they lie downstream of it.

✏️ **Your turn.** (a) Add the edge P → B ("the police scare off burglars"). Is the graph still a DAG? Which nodes are
on the cycle? (b) Add J → M ("John tells Mary") instead. Allowed? Which new topological order do you get? Predict
first, then run `topological_order` and `on_a_cycle`.

### 📑 Slide connection — `008_BN_mz_021825`, slides 9–12

- **What the slides show.** The alarm example with six events (Burglary, Earthquake, Alarm, John calls, Mary calls, Police come);
    a DAG is a set of nodes plus directed edges with no cycles; B → E would be OK, M → E is "Not ok" (cycle);
    terminology: parents of A = {B, E}, children of A = {J, M}, descendants of A = {J, M, P}.
- **What this notebook adds.** Code for parents/children/descendants/ancestors; Kahn's topological sort as a cycle detector (the
    slides state the rule, here you can test any graph); why cycles are forbidden; the biological price (no
    feedback loops) and the course's two escapes (DBNs, GGMs).
- **Exam tip.** "Descendants" includes grandchildren (P is a descendant of A, not a child). Descendants matter in
    Module 16: a collider is unblocked if the collider *or any of its descendants* is observed.

---
## 3 · The joint distribution of a Bayesian network

> 📍 **Where we are.** We have the graph. The second ingredient is the probability part: each node gets a
> **conditional probability distribution** (CPD) given its parents (slide 13). Multiplying them gives the joint
> distribution of all variables (slides 14–16).

For binary variables a CPD is a small table, a **conditional probability table** (CPT), with one row per combination
of parent values. A node with $k$ binary parents has $2^k$ rows, and each row holds one free number,
$P(X=1\mid\text{that parent combination})$. Here are the alarm CPTs with the values of slide 99 (we use only the five
nodes B, E, A, J, M from now on; the police node P appears only on slides 9–16).

In [ ]:
# ── 3.1 The conditional probability tables of the alarm network ──
def cpt_table(node, parents=ALARM_PARENTS, cpts=ALARM_CPTS):
    """An ALARM-style CPT as a readable pandas table (one row per parent configuration)."""
    rows = []
    for pa_vals, p1 in cpts[node].items():
        row = dict(zip(parents[node], pa_vals))
        row[f"P({node}=1|·)"] = p1
        row[f"P({node}=0|·)"] = round(1 - p1, 10)
        rows.append(row)
    return pd.DataFrame(rows)


for node in ALARM_PARENTS:
    print(f"── {node}   parents: {', '.join(ALARM_PARENTS[node]) or 'none'}")
    print(cpt_table(node).to_string(index=False), "\n")

Each row of a CPT sums to 1 (that is why only $P(\cdot=1\mid\cdot)$ is stored). Reading the table for A: a burglary
without an earthquake sets off the alarm with probability 0.85; an earthquake alone with probability 0.5; neither,
with probability 0.05 (false alarms). Mary is the more eager caller: she phones for 80 % of real alarms but also
15 % of the time when there is no alarm at all ($P(M{=}1\mid \neg A)=0.15$); John is more reserved — he misses 30 % of
real alarms but rarely calls without one ($P(J{=}1\mid \neg A)=0.05$).

**The key formula (slide 15).** A Bayesian network with nodes $X_1,\dots,X_n$ defines the joint distribution

$$P(X_1,\dots,X_n) = \prod_{i=1}^{n} P\big(X_i \mid \mathrm{Pa}(X_i)\big).$$

*In words:* the probability of a complete configuration of all variables is the product, over every node, of the
probability of that node's value given the values of its parents. Roots (no parents) contribute their plain
probability. For the alarm network:

$$P(B,E,A,J,M) = P(B)\,P(E)\,P(A\mid B,E)\,P(J\mid A)\,P(M\mid A).$$

*In words:* draw burglary and earthquake independently, then the alarm given both, then each neighbour's call given
only the alarm. With the police node the slides add one more factor, $P(P\mid J,M)$.

In [ ]:
# ── 3.2 The joint probability of one complete configuration (slides 98–101) ──
def cpt_prob(node, value, assignment, parents, cpts):
    """P(node = value | the values its parents take in `assignment`), from an ALARM-style CPT."""
    pa_vals = tuple(assignment[p] for p in parents[node])
    p1 = cpts[node][pa_vals]
    return p1 if value == 1 else 1 - p1


def joint_prob(assignment, parents, cpts):
    """P(X1 = x1, ..., Xn = xn) = product over nodes of P(x_i | pa(x_i))."""
    prob = 1.0
    for node in parents:
        prob *= cpt_prob(node, assignment[node], assignment, parents, cpts)
    return prob


world = {"B": 1, "E": 0, "A": 1, "J": 1, "M": 0}       # burglary, no earthquake, alarm, John calls, Mary does not
terms = []
for node in ALARM_PARENTS:
    pa = ALARM_PARENTS[node]
    cond = "|" + ",".join(f"{p}={world[p]}" for p in pa) if pa else ""
    terms.append((f"P({node}={world[node]}{cond})", cpt_prob(node, world[node], world, ALARM_PARENTS, ALARM_CPTS)))
print(" × ".join(name for name, _ in terms))
print(" = " + " × ".join(f"{v:g}" for _, v in terms))
print(f" = {joint_prob(world, ALARM_PARENTS, ALARM_CPTS):.6f}")

🔍 **What just happened?** We reproduced slide 101 exactly:
$P(B,\neg E,A,J,\neg M)=0.05\times0.9\times0.85\times0.7\times0.2=0.005355$. Note the two "not" factors:
$P(\neg E)=1-0.1=0.9$ and $P(\neg M\mid A)=1-0.8=0.2$. Five table look-ups and four multiplications — computing a
*complete* configuration is always this cheap. (Questions about *some* of the variables, like "John called, was
there a burglary?", need sums over the others; that is Module 17.)

Now list **all** $2^5=32$ configurations ("possible worlds") with their probabilities.

In [ ]:
# ── 3.3 The full joint table: 32 possible worlds ──
def full_joint(parents, cpts):
    """Every assignment of a binary BN with its probability, as a DataFrame with a column 'prob'."""
    nodes = list(parents)
    rows = []
    for values in itertools.product([0, 1], repeat=len(nodes)):
        assignment = dict(zip(nodes, values))
        rows.append({**assignment, "prob": joint_prob(assignment, parents, cpts)})
    return pd.DataFrame(rows)


alarm_table = full_joint(ALARM_PARENTS, ALARM_CPTS)
print(f"{len(alarm_table)} rows; probabilities sum to {alarm_table['prob'].sum():.12f}")
ranked = alarm_table.sort_values("prob", ascending=False).reset_index(drop=True)
print("\nthe six most probable worlds:")
print(ranked.head(6).to_string())
rank = ranked.index[(ranked[list(world)] == pd.Series(world)).all(axis=1)][0]
print(f"\nthe slide-101 world (B=1,E=0,A=1,J=1,M=0) is number {rank + 1} of 32")

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(np.arange(1, 33), ranked["prob"], color="tab:blue")
ax.bar(rank + 1, ranked["prob"][rank], color="tab:orange", label="B, ¬E, A, J, ¬M (slide 101)")
ax.set_yscale("log"); ax.set_xlabel("world, ranked by probability"); ax.set_ylabel("P(world) (log scale)")
ax.set_title("the 32 joint probabilities of the alarm network"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 32 probabilities add up to exactly 1, as a distribution must. The single most probable
world is "nothing happens and nobody calls" (0.656); the runner-up is "nothing happens but Mary calls anyway"
(0.116), followed by a silent earthquake (0.038) and a false call from John (0.035) — false calls are far more common
than burglaries. The slide-101 world ranks 16th of 32. The log scale shows probabilities spanning more than five
orders of magnitude.

**Why does the product always sum to 1?** Sum over the values of a **leaf** first, say $M$. Only one factor contains
$M$, and $\sum_m P(M{=}m\mid A)=1$ because each CPT row sums to 1:

$$\sum_{m} P(B)P(E)P(A\mid B,E)P(J\mid A)P(m\mid A) = P(B)P(E)P(A\mid B,E)P(J\mid A)\underbrace{\sum_m P(m\mid A)}_{=1}.$$

*In words:* summing out a leaf simply deletes its factor. Repeat with the new leaves ($J$, then $A$, then $E$, then
$B$) and you are left with 1. The same argument shows that the marginal distribution of the remaining variables is
again the product of *their* factors — let us verify that numerically.

In [ ]:
# ── 3.4 Summing out a leaf deletes its factor ──
marg_BEAJ = alarm_table.groupby(["B", "E", "A", "J"])["prob"].sum()            # sum over M
parents_no_M = {k: v for k, v in ALARM_PARENTS.items() if k != "M"}
product_no_M = pd.Series({idx: joint_prob(dict(zip(["B", "E", "A", "J"], idx)), parents_no_M, ALARM_CPTS)
                          for idx in marg_BEAJ.index})
print(f"max |Σ_M P(B,E,A,J,M) − P(B)P(E)P(A|B,E)P(J|A)| = {np.max(np.abs(marg_BEAJ - product_no_M)):.2e}")

🔍 **What just happened?** The difference is at the level of floating-point round-off ($\sim10^{-17}$): marginalizing
the leaf $M$ really does just remove $P(M\mid A)$. This "peeling leaves" trick is also the first step of the
inference algorithms of Module 17.

**Constructing a Bayesian network (slide 17).**
1. **Identify the random variables** (and their possible values).
2. **Determine the graph** — the direct dependencies, usually from domain knowledge.
3. **Determine the conditional probability distribution** of each node given its parents (from knowledge, or
   learned from data — Module 17).

✏️ **Your turn.** Imagine a high-crime neighbourhood: set `P(B=1) = 0.5` (copy `ALARM_CPTS`, change one number, and
rebuild the table with `full_joint`). Which world is now the most probable? Does the table still sum to 1? Why did you
not need to change any other number to keep it a valid distribution?

### 📑 Slide connection — `008_BN_mz_021825`, slides 13–17

- **What the slides show.** Nodes are random variables, edges probabilistic dependencies, and each node carries a conditional
    probability distribution given its parents, $P(X_i\mid \mathrm{Pa}(X_i))$; the BN defines the joint distribution
    $P(X_1\dots X_n)=\prod_i P(X_i\mid \mathrm{Pa}(X_i))$, e.g. $P(B,E,A,J,M,P)=P(B)P(E)P(A\mid B,E)P(J\mid A)P(M\mid A)
    P(P\mid J,M)$; summary: identify variables, determine the graph, determine the CPDs.
- **What this notebook adds.** The CPTs printed as tables; a general `joint_prob` that reproduces 0.005355 term by term (slides 98–101);
    the full 32-world table and its ranking; a proof (and a numerical check) that the product sums to 1 because
    summing out a leaf deletes its factor.
- **Exam tip.** Watch the negations: $P(\neg E)=1-P(E)$ and $P(\neg M\mid A)=1-P(M\mid A)$. A common exam slip is to use
    $P(M\mid\neg A)=0.15$ instead of $P(\neg M\mid A)=0.2$.

---
## 4 · Factorising joint distributions: 31 versus 10 parameters

> 📍 **Where we are.** Section 3 showed *how* a Bayesian network defines a joint distribution. This section shows
> *why* that is worth doing: the factorization needs far fewer numbers than the full table (slides 18–29). This is the
> course's second recurring idea — **conditional independence buys tractability**.

**The chain rule needs no assumptions (slide 18).** For the five binary variables, apply $P(X,Y)=P(X\mid Y)P(Y)$ four
times:

$$\begin{aligned}
P(M,J,A,B,E) &= P(M\mid J,A,B,E)\,P(J,A,B,E)\\
&= P(M\mid J,A,B,E)\,P(J\mid A,B,E)\,P(A,B,E)\\
&= P(M\mid J,A,B,E)\,P(J\mid A,B,E)\,P(A\mid B,E)\,P(B\mid E)\,P(E).
\end{aligned}$$

*In words:* peel off one variable at a time, each time conditioning it on all the variables still remaining. This is
always true, for any distribution — it is just the definition of conditional probability.

Read it as a Bayesian network (slide 19): each factor's conditioning variables are the node's parents. $M$ has
parents $J, A, B, E$; $J$ has $A, B, E$; and so on. Every pair of nodes is joined by an arrow — a **complete DAG**.

In [ ]:
# ── 4.1 The chain rule is a complete DAG (slides 19 and 29) ──
CHAIN_PARENTS = {"E": (), "B": ("E",), "A": ("B", "E"), "J": ("A", "B", "E"), "M": ("J", "A", "B", "E")}
CHAIN_POS = {"A": (0.15, 0.85), "B": (0.6, 0.85), "E": (0.9, 0.5), "J": (0.15, 0.12), "M": (0.6, 0.12)}


def edges_from_parents(parents):
    """Convert a parents dictionary into a list of (parent, child) edges."""
    return [(p, child) for child, pa in parents.items() for p in pa]


fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
draw_graph(edges_from_parents(CHAIN_PARENTS), CHAIN_POS, ax=axes[0],
           title=f"chain rule: complete DAG ({len(edges_from_parents(CHAIN_PARENTS))} edges)")
draw_graph(edges_from_parents(ALARM_PARENTS), ALARM_POS, ax=axes[1],
           title=f"alarm network: domain knowledge ({len(edges_from_parents(ALARM_PARENTS))} edges)")
plt.tight_layout(); plt.show()

**Counting free parameters.** A node $X_i$ with $r_i$ possible values and parents with $r_{p}$ values each has a CPT
with one row per parent configuration and $r_i-1$ free numbers per row:

$$\#\text{free parameters of } X_i = (r_i - 1)\prod_{p\in \mathrm{Pa}(X_i)} r_p .$$

*In words:* (number of values of the node, minus one because each row sums to 1) times (number of rows = number of
combinations of parent values). For binary variables this is $2^{k}$ for a node with $k$ parents. The total for the
network is the sum over nodes.

In [ ]:
# ── 4.2 A parameter counter for any DAG (slides 23 and 28) ──
def count_free_params(parents, card):
    """Free parameters of each node's CPT: (r_i − 1) × product of the parents' numbers of values."""
    return {node: (card[node] - 1) * int(np.prod([card[p] for p in pa])) for node, pa in parents.items()}


binary = {v: 2 for v in "BEAJM"}
for name, parents in [("chain rule (complete DAG)", CHAIN_PARENTS), ("alarm network", ALARM_PARENTS)]:
    counts = count_free_params(parents, binary)
    detail = "  ".join(f"P({v}|{','.join(parents[v])})={c}" if parents[v] else f"P({v})={c}"
                       for v, c in counts.items())
    print(f"{name:26s} {detail}   total = {sum(counts.values())}")
print(f"full joint table: 2^5 − 1 = {2**5 - 1}")

🔍 **What just happened?** The chain rule needs $16+8+4+2+1=31=2^5-1$ numbers — exactly as many as the full table,
because it *is* the full table, rewritten. The alarm network needs $1+1+4+2+2=10$. "By relying on domain knowledge we
saved 21 parameters!" (slide 28).

Where did the savings come from? From **deleting arrows**, i.e. from assumptions:
- *Is E causal for B?* (slide 24) No — earthquakes do not cause burglaries, so drop E → B: $P(B\mid E)=P(B)$.
- John reacts only to the alarm, not to what caused it: $P(J\mid A,B,E)=P(J\mid A)$.
- Mary ignores John and the causes: $P(M\mid J,A,B,E)=P(M\mid A)$.

⚠️ **The complete DAG can represent any distribution; a sparse DAG cannot.** Every missing arrow is a claim about
the world (a conditional independence, Module 16). If the claim is wrong, the model is wrong, however many data you
have. Conversely, if the claims are true, the 31-number description must contain only 10 distinct numbers. Let us
check that in the alarm distribution.

In [ ]:
# ── 4.3 The chain-rule CPTs of the alarm distribution contain only 10 distinct numbers ──
def conditional_from_table(table, node, given):
    """P(node = 1 | each configuration of `given`), computed from a full joint table."""
    num = table[table[node] == 1].groupby(list(given))["prob"].sum()
    den = table.groupby(list(given))["prob"].sum()
    return (num / den).round(10)


for node, given in [("M", ["J", "A", "B", "E"]), ("J", ["A", "B", "E"]), ("A", ["B", "E"]), ("B", ["E"])]:
    cond = conditional_from_table(alarm_table, node, given)
    print(f"P({node}=1 | {','.join(given)}): {len(cond):2d} rows, distinct values {sorted(set(cond))}")

🔍 **What just happened?** We computed the chain-rule CPTs directly from the 32-row joint table of Section 3. The 16
rows of $P(M\mid J,A,B,E)$ take only the two values 0.15 and 0.8 — the value depends on $A$ alone. The 8 rows of
$P(J\mid A,B,E)$ take only 0.05 and 0.7, and $P(B\mid E)$ is 0.05 whatever $E$ is. The full 31-number description of
this distribution is redundant, and the Bayesian network stores only the distinct numbers.

### 4.1 Scaling up: gene regulatory networks

Five variables is a toy. What happens for a regulatory network of $n$ genes in which every gene has **at most three
regulators**? Each binary gene then needs at most $2^3=8$ numbers, so at most $8n$ in total; with three expression
levels, at most $2\cdot3^3=54$ per gene.

In [ ]:
# ── 4.4 Exponential versus linear: full joint vs a network with ≤ 3 parents per gene ──
n = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(n, 2.0**n - 1, color="tab:red", label="full joint, binary: $2^n-1$")
ax.plot(n, 3.0**n - 1, color="tab:red", ls="--", label="full joint, 3 levels: $3^n-1$")
ax.plot(n, 8 * n, color="tab:blue", label="BN, ≤ 3 parents, binary: $8n$")
ax.plot(n, 54 * n, color="tab:blue", ls="--", label="BN, ≤ 3 parents, 3 levels: $54n$")
ax.set_yscale("log"); ax.set_xlabel("number of genes n"); ax.set_ylabel("free parameters (log scale)")
ax.set_title("sparse graphs turn exponential growth into linear growth"); ax.legend()
plt.tight_layout(); plt.show()
print(f"n = 30 binary genes: full joint {2**30 - 1:,} vs Bayesian network ≤ {8 * 30}")

🔍 **What just happened?** On a log scale the full-joint curves are straight lines rising steeply (exponential
growth); the network curves barely move (linear growth). For 30 binary genes the full table needs 1,073,741,823
numbers, a network with at most three regulators per gene at most 240. Biology helps: most genes are regulated by a
handful of transcription factors, so sparse networks are a reasonable starting assumption.

### 4.2 A real curated network: the Sachs signalling network

The toolkit holds the 20-edge **consensus network** of the 11 proteins measured by Sachs *et al.* (2005) — the
reference network used in Modules 16–18. Is it a DAG, and how many parameters does it need if each protein is
recorded as low/medium/high?

In [ ]:
# ── 4.5 The Sachs et al. (2005) consensus network: a DAG with 230 parameters ──
SACHS_POS = {"plc": (0.08, 0.88), "pip3": (0.42, 0.97), "pip2": (0.08, 0.5), "pkc": (0.4, 0.68),
             "pka": (0.62, 0.42), "raf": (0.15, 0.3), "mek": (0.3, 0.05), "erk": (0.65, 0.05),
             "akt": (0.95, 0.2), "p38": (0.8, 0.8), "jnk": (0.97, 0.55)}
sachs_parents = {p: tuple(a for a, b in SACHS_CONSENSUS_EDGES if b == p) for p in SACHS_PROTEINS}
order, leftover = topological_order(SACHS_PROTEINS, SACHS_CONSENSUS_EDGES)
print("topological order:", order, "| leftover:", leftover)
counts3 = count_free_params(sachs_parents, {p: 3 for p in SACHS_PROTEINS})
print("parameters per protein (3 levels):", counts3)
print(f"total = {sum(counts3.values())}   vs   full joint 3^11 − 1 = {3**11 - 1:,}")

fig, ax = plt.subplots(figsize=(5.2, 4.6))
draw_graph(SACHS_CONSENSUS_EDGES, SACHS_POS, ax=ax, radius=0.055,
           title="consensus signalling network (Sachs et al. 2005)\n11 proteins, 20 directed edges")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The consensus network is acyclic (nothing left over), with PIP3 as the only root. Each
protein has at most three parents (Mek and Akt), so the network needs $2\cdot3^{|\mathrm{Pa}|}$ numbers per protein:
**230 in total instead of 177,146** — a 770-fold reduction. With 1,800 measured cells (Module 17), 230 numbers are
learnable; 177,146 are not.

✏️ **Your turn.** (a) Recount with five levels per protein (`{p: 5 for p in SACHS_PROTEINS}`) and compare with
$5^{11}-1$. (b) Which single extra edge would add the most parameters? (Hint: an edge into a node that already has
three parents multiplies its table by 3.)

### 📑 Slide connection — `008_BN_mz_021825`, slides 18–29

- **What the slides show.** The chain rule $P(M,J,A,B,E)=P(M\mid J,A,B,E)P(J\mid A,B,E)P(A\mid B,E)P(B\mid E)P(E)$ assumes no
    structure; its DAG (slides 19–23) is complete and needs 16 + 8 + 4 + 2 + 1 = 31 = $2^5-1$ parameters
    ("exponential complexity"); asking "Is E causal for B?" and using domain knowledge gives
    $P(B)P(E)P(A\mid B,E)P(J\mid A)P(M\mid A)$ with 1 + 1 + 4 + 2 + 2 = 10 parameters — 21 saved (slides 25–29).
- **What this notebook adds.** A general parameter counter for nodes with any number of values; a numerical demonstration that the
    alarm distribution's chain-rule CPTs contain only 10 distinct numbers; the scaling curves for gene networks; the
    real Sachs consensus network (230 vs 177,146 parameters) checked for acyclicity.
- **Exam tip.** The count for a binary node is $2^{\#\text{parents}}$ (one number per row), not $2^{\#\text{parents}+1}$.
    For non-binary nodes use $(r-1)\times\prod r_{\text{parents}}$. Fewer parameters is not free: every deleted edge is an
    independence assumption that could be wrong.

---
## 5 · Numerical example: movie preference

> 📍 **Where we are.** We now run the three-step recipe of slide 17 on the small example of slides 30–36, compute a
> joint probability by hand, and then use the full table to answer questions the slides leave for later.

1. **Variables.** $Le$: the movie is long; $Li$: you like the movie; $S$: you sleep during the movie (all binary).
2. **Graph.** A long movie makes you less likely to like it and more likely to sleep: $Le\to Li$ and $Le\to S$.
3. **CPDs** (slides 32–34): $P(Le{=}1)=0.5$; $P(Li{=}1\mid Le{=}1)=0.4$, $P(Li{=}1\mid Le{=}0)=0.7$;
   $P(S{=}1\mid Le{=}1)=0.6$, $P(S{=}1\mid Le{=}0)=0.2$.

The joint distribution (slide 35) is

$$P(Le, Li, S) = P(Le)\,P(Li\mid Le)\,P(S\mid Le).$$

*In words:* pick the movie length, then — independently of each other, given the length — whether you like it and
whether you fall asleep.

In [ ]:
# ── 5.1 The movie network and its joint table (slides 30–36) ──
MOVIE_PARENTS = {"Le": (), "Li": ("Le",), "S": ("Le",)}
MOVIE_CPTS = {"Le": {(): 0.5}, "Li": {(1,): 0.4, (0,): 0.7}, "S": {(1,): 0.6, (0,): 0.2}}
movie_table = full_joint(MOVIE_PARENTS, MOVIE_CPTS)
p_111 = joint_prob({"Le": 1, "Li": 1, "S": 1}, MOVIE_PARENTS, MOVIE_CPTS)
print(f"P(Le=1, Li=1, S=1) = 0.5 × 0.4 × 0.6 = {p_111:.2f}")
print(movie_table.to_string(index=False))
print(f"sum = {movie_table['prob'].sum():.6f}; free parameters: network "
      f"{sum(count_free_params(MOVIE_PARENTS, {'Le': 2, 'Li': 2, 'S': 2}).values())} vs full table {2**3 - 1}")

🔍 **What just happened?** $P(Le{=}1,Li{=}1,S{=}1)=0.5\times0.4\times0.6=0.12$, as on slide 36: a long movie that
you liked *and* slept through happens 12 % of the time. The eight probabilities sum to 1, and the network needs 5
numbers instead of 7.

The slides stop at the joint probability. But the joint table contains the answer to **every** question about these
three variables. Questions with a bar ("given") are answered by the definition of conditional probability: add up
the rows where everything holds, and divide by the sum of the rows where the condition holds.

In [ ]:
# ── 5.2 Answering questions from the joint table ──
def prob(table, event, given=None):
    """P(event | given) from a joint table; `event` and `given` are dicts {variable: value}."""
    given = given or {}

    def rows(cond):
        mask = np.ones(len(table), bool)
        for var, val in cond.items():
            mask &= table[var].to_numpy() == val
        return mask

    return table.loc[rows({**event, **given}), "prob"].sum() / table.loc[rows(given), "prob"].sum()


print(f"P(S=1)                = {prob(movie_table, {'S': 1}):.3f}")
print(f"P(Le=1 | S=1)         = {prob(movie_table, {'Le': 1}, {'S': 1}):.3f}   (falling asleep is evidence of a long movie)")
print(f"P(Li=1)               = {prob(movie_table, {'Li': 1}):.3f}")
print(f"P(Li=1 | S=1)         = {prob(movie_table, {'Li': 1}, {'S': 1}):.3f}   (sleeping lowers the chance you liked it)")
print(f"P(Li=1 | S=1, Le=1)   = {prob(movie_table, {'Li': 1}, {'S': 1, 'Le': 1}):.3f}")
print(f"P(Li=1 | S=0, Le=1)   = {prob(movie_table, {'Li': 1}, {'S': 0, 'Le': 1}):.3f}   (= P(Li=1 | Le=1): S adds nothing)")

🔍 **What just happened?** $P(S{=}1)=0.5\cdot0.6+0.5\cdot0.2=0.4$. Learning that someone slept raises the probability
that the movie was long from 0.5 to $0.3/0.4=0.75$ (Bayes' rule) — and, *through* the length, lowers the chance they
liked it from 0.55 to 0.475. But once we know the length, sleeping says nothing more about liking:
$P(Li{=}1\mid S, Le{=}1)=0.4$ whether $S$ is 0 or 1. Same structure, same behaviour as smoking/nails/cancer in
Section 1: $Li\perp S\mid Le$, a **common cause** screens off its effects.

✏️ **Your turn — the biology version.** Rename $Le$ → "transcription factor active", $Li$ → "target gene 1 on",
$S$ → "target gene 2 on" and choose your own CPT values (e.g. each target is on with probability 0.9 when the factor
is active and 0.1 otherwise). Compute $P(\text{gene 2 on}\mid\text{gene 1 on})$ and
$P(\text{gene 2 on}\mid\text{gene 1 on},\text{TF active})$. Two co-expressed genes are not necessarily regulating
each other — this is Example 15 of Module 16.

### 📑 Slide connection — `008_BN_mz_021825`, slides 30–37

- **What the slides show.** Movie preference in three steps: variables Le, Li, S; graph Le → Li, Le → S ("long movie → don't like",
    "long movie → sleep"); CPDs P(Le=1) = 0.5, P(Li=1|Le=1) = 0.4, P(Li=1|Le=0) = 0.7, P(S=1|Le=1) = 0.6,
    P(S=1|Le=0) = 0.2; joint $P(Le,Li,S)=P(Le)P(Li\mid Le)P(S\mid Le)$ and
    $P(Le{=}1,Li{=}1,S{=}1)=0.5\times0.4\times0.6$. Summary (slide 37): DAG + joint as a product over parents —
    "more intuitive representation, fewer parameters".
- **What this notebook adds.** The whole 8-row table; a general `prob(table, event, given)` that answers any question from a joint
    table; the numbers behind "sleeping is evidence of a long movie" (0.75) and "given the length, sleeping says
    nothing about liking" (0.4 either way) — a preview of Modules 16 and 17.
- **Exam tip.** The slide's arithmetic stops at $0.5\times0.4\times0.6$; the answer is 0.12. Be ready to compute a marginal
    such as $P(S{=}1)=\sum_{le}P(le)P(S{=}1\mid le)=0.4$ and a posterior such as $P(Le{=}1\mid S{=}1)=0.75$ by hand.

---
## 6 · A reusable Bayesian-network class

> 📍 **Where we are.** We have written `topological_order`, `joint_prob` and `full_joint` separately. Here we
> package them, plus a sampler, into a 40-line class. Modules 16 and 17 use the same functions (re-defined so that
> each notebook runs on its own).

**Ancestral (forward) sampling.** A Bayesian network is also a *recipe for generating data*: visit the nodes in
topological order and draw each one given the values already drawn for its parents. To draw a binary value with
$P(X{=}1\mid\text{parents})=\theta$, take a uniform random number $u$ between 0 and 1 and set

$$X = \begin{cases}1 & \text{if } u < \theta\\ 0 & \text{otherwise.}\end{cases}$$

*In words:* $u$ falls below $\theta$ exactly a fraction $\theta$ of the time, so $X$ is 1 with probability $\theta$.
Because every node is drawn from its correct conditional given its parents, the full sample follows the product
formula of Section 3. Module 17 uses these samples for approximate inference.

In [ ]:
# ── 6.1 A small Bayesian-network class (binary variables, ALARM-style CPTs) ──
class DiscreteBN:
    """Binary Bayesian network: a parents dict + CPTs storing P(node = 1 | parent values)."""

    def __init__(self, parents, cpts):
        self.parents, self.cpts = dict(parents), dict(cpts)
        nodes = list(self.parents)
        self.order, leftover = topological_order(nodes, edges_from_parents(self.parents))
        if leftover:
            raise ValueError(f"not a DAG: {leftover} are on (or below) a cycle")
        for node in nodes:                   # one CPT row per parent configuration
            if len(self.cpts[node]) != 2 ** len(self.parents[node]):
                raise ValueError(f"CPT of {node} needs {2 ** len(self.parents[node])} rows")

    def n_params(self):
        """Free parameters: one number per CPT row."""
        return sum(len(table) for table in self.cpts.values())

    def joint(self, assignment):
        """Probability of one complete assignment {node: 0/1}."""
        return joint_prob(assignment, self.parents, self.cpts)

    def table(self):
        """All 2^n assignments with their probabilities."""
        return full_joint(self.parents, self.cpts)

    def sample(self, n, rng):
        """Ancestral sampling of n complete assignments (one row each)."""
        out = {}
        for node in self.order:                               # parents are always drawn first
            p1 = np.empty(n)
            for pa_vals, theta in self.cpts[node].items():    # look up θ for every sample's parent values
                rows = np.ones(n, bool)
                for parent, value in zip(self.parents[node], pa_vals):
                    rows &= out[parent] == value
                p1[rows] = theta
            out[node] = (rng.random(n) < p1).astype(int)
        return pd.DataFrame(out)[list(self.parents)]

In [ ]:
# ── 6.2 The alarm network as an object ──
alarm = DiscreteBN(ALARM_PARENTS, ALARM_CPTS)
print("topological order :", alarm.order)
print("free parameters   :", alarm.n_params())
print(f"P(B,¬E,A,J,¬M)    : {alarm.joint(world):.6f}")
tab = alarm.table()
p_alarm = tab.loc[tab["A"] == 1, "prob"].sum()
print(f"P(A=1) = Σ over the other variables = {p_alarm:.6f}")
try:
    DiscreteBN({**ALARM_PARENTS, "E": ("M",)}, {**ALARM_CPTS, "E": {(1,): 0.1, (0,): 0.1}})
except ValueError as err:
    print("adding M→E:", err)

🔍 **What just happened?** The object knows its topological order (B, E, A, J, M), its 10 parameters and reproduces
0.005355. Summing the table gives the overall probability that the alarm rings on a given day,
$P(A{=}1)=0.13325$ — about one day in 7.5, made up in roughly equal parts of burglaries (0.043), earthquakes without
a burglary (0.048) and false alarms (0.043); with these toy numbers earthquakes are very frequent. The constructor
refuses the cyclic graph of slide 11.

Now generate 100,000 "days" by ancestral sampling and compare the observed frequency of each of the 32 worlds with its
exact probability.

In [ ]:
# ── 6.3 Ancestral sampling reproduces the joint distribution ──
rng = np.random.default_rng(SEED + 6)
days = alarm.sample(100_000, rng)
print(days.head(3).to_string(), "\n")
freq = days.value_counts(normalize=True).rename("freq").reset_index()
compare = tab.merge(freq, on=list(ALARM_PARENTS), how="left").fillna({"freq": 0.0})
print(f"P(B=1): sampled {days['B'].mean():.4f} vs exact 0.0500 | P(A=1): sampled {days['A'].mean():.4f} vs exact {p_alarm:.4f}")
rarest = compare.loc[compare["prob"].idxmin()]
rarest_name = ",".join(f"{v}={int(rarest[v])}" for v in ALARM_PARENTS)
print(f"worlds never sampled: {(compare['freq'] == 0).sum()} of 32 | rarest world ({rarest_name}): "
      f"exact {rarest['prob']:.1e} (expected count {rarest['prob'] * len(days):.2f}), observed {round(rarest['freq'] * len(days))}")

fig, ax = plt.subplots(figsize=(4.6, 4))
seen = compare["freq"] > 0
ax.loglog(compare.loc[seen, "prob"], compare.loc[seen, "freq"], "o", color="tab:blue", ms=5, label="sampled worlds")
ax.plot([1e-6, 1], [1e-6, 1], "k--", lw=1, label="frequency = probability")
ax.set_xlabel("exact probability of the world"); ax.set_ylabel("frequency in 100,000 samples")
ax.set_title("ancestral sampling vs exact joint (32 worlds)"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each sampled row is one simulated day, and the sampled $P(B{=}1)=0.0492$ and
$P(A{=}1)=0.1317$ are close to the exact 0.05 and 0.1333. Frequent worlds land right on the diagonal; rare worlds
(expected counts of a few to a few dozen) scatter around it. All 32 worlds were seen at least once — even the rarest,
"burglary *and* earthquake, yet no alarm, and both neighbours call" (exact probability $1.9\times10^{-6}$, expected
0.19 times in 100,000 days), which appeared once by luck: its frequency $10^{-5}$ is five times too high (the lonely dot
at the bottom left). This is the basic trade-off of sampling methods (Module 17): cheap, but rare events need many
samples.

✏️ **Your turn.** Re-run with 1,000 samples instead of 100,000. How many worlds are never seen? Would you trust a
frequency-based estimate of $P(B{=}1\mid J{=}1, M{=}1)$ from 1,000 samples? (Count how many sampled days have
$J=M=1$.)

Finally, the sixth node of slides 9–16. Adding the police node P with parents J and M needs a CPT with four rows; the
slides give no numbers, so we choose **illustrative** ones.

In [ ]:
# ── 6.4 Adding a leaf (the police, slides 9–16) leaves the rest of the network unchanged ──
POLICE_CPT = {(1, 1): 0.9, (1, 0): 0.6, (0, 1): 0.6, (0, 0): 0.01}      # illustrative, not from the slides
alarm6 = DiscreteBN({**ALARM_PARENTS, "P": ("J", "M")}, {**ALARM_CPTS, "P": POLICE_CPT})
tab6 = alarm6.table()
marg5 = tab6.groupby(list(ALARM_PARENTS))["prob"].sum().reset_index()
diff = np.max(np.abs(marg5["prob"].to_numpy() - tab.sort_values(list(ALARM_PARENTS))["prob"].to_numpy()))
print(f"6-node network: {alarm6.n_params()} parameters, order {alarm6.order}")
print(f"max |Σ_P P(B,E,A,J,M,P) − P(B,E,A,J,M)| = {diff:.1e}")
print(f"P(police come) = P(P=1) = {tab6.loc[tab6['P'] == 1, 'prob'].sum():.4f}")

🔍 **What just happened?** The six-node network needs 10 + 4 = 14 parameters. Summing out the new leaf P gives back
exactly the five-node distribution (difference $\sim10^{-17}$, the "peeling" argument of Section 3 again): adding
downstream variables never changes what the model says about upstream ones. (With our illustrative CPT the police
come on about 21 % of days — mostly because of false calls; the number means nothing beyond this toy.)

---
## 7 · Bayesian networks in biology

> 📍 **Where we are.** Everything so far used burglars and movies. This section shows two biological Bayesian
> networks: one where the CPTs are **known from a mechanism** (Mendel's laws), and one where they must be **learned
> from data** (signalling networks; Module 17). It ends with what an edge does — and does not — mean.

### 7.1 Mendelian inheritance is a Bayesian network

Mother → child ← father. Code genotypes as the number of copies of allele T (AA = 0, AT = 1, TT = 2, the course's
convention). If the T allele has frequency $q$ and parents are drawn from a population in Hardy–Weinberg equilibrium,

$$P(\text{parent}=0,1,2) = \big((1-q)^2,\; 2q(1-q),\; q^2\big).$$

*In words:* a parent is AA if both of its alleles are A, TT if both are T, and AT in the two remaining ways.
The child's CPT comes from Mendel's laws: a parent with genotype $g$ transmits T with probability $t_g = g/2$
(0, ½ or 1), independently for mother and father.

In [ ]:
# ── 7.1 A trio pedigree as a Bayesian network with a CPT fixed by Mendel's laws ──
q = 0.3                                                  # frequency of allele T
p_parent = np.array([(1 - q)**2, 2 * q * (1 - q), q**2])   # Hardy–Weinberg: P(genotype 0, 1, 2)
t = np.array([0.0, 0.5, 1.0])                            # P(transmit T | parent genotype 0, 1, 2)
cpt_kid = np.zeros((3, 3, 3))                            # cpt_kid[m, d, k] = P(kid = k | mom = m, dad = d)
for m in range(3):
    for d in range(3):
        cpt_kid[m, d] = [(1 - t[m]) * (1 - t[d]), t[m] * (1 - t[d]) + (1 - t[m]) * t[d], t[m] * t[d]]
print("P(kid = 0/1/2 | AT × AT) =", cpt_kid[1, 1], "  every CPT row sums to 1:", np.allclose(cpt_kid.sum(-1), 1))

trio = p_parent[:, None, None] * p_parent[None, :, None] * cpt_kid        # trio[m, d, k] = joint probability
print("P(kid genotype) =", trio.sum(axis=(0, 1)).round(4), "  Hardy–Weinberg:", p_parent.round(4))
post_dad = trio[0, :, 1] / trio[0, :, 1].sum()            # P(dad | mom = AA, kid = AT)
print("P(dad = 0/1/2 | mom = AA, kid = AT) =", post_dad.round(3))
print(f"free parameters: CPTs learned from data would need 2 + 2 + 9×2 = {2 + 2 + 18}; with Mendel's laws only q (1)")

🔍 **What just happened?** The AT × AT row is Mendel's 1 : 2 : 1 ratio. The child's marginal distribution equals the
parents' Hardy–Weinberg distribution — the network reproduces the textbook fact that random mating preserves HWE.
And we inferred something hidden: if the mother is AA and the child AT, the child's T came from the father, who is
AT with probability 0.7 and TT with probability 0.3 (= $q$; can you see why?). Pedigree analysis and genetic
counselling software are, at heart, Bayesian networks of this kind; here domain knowledge fixes the CPT entirely,
leaving a single parameter $q$.

✏️ **Your turn.** Make the T allele rare (`q = 0.05`) and re-run. What is now $P(\text{dad}=\text{TT}\mid
\text{mom}=\text{AA},\text{kid}=\text{AT})$? Then compute $P(\text{mom}=\text{TT}\mid\text{kid}=\text{TT})$ from `trio`
(sum over the father) and explain why it equals $q$.

### 7.2 Gene regulatory and signalling networks

Slides 6 and 58 draw gene regulatory networks: transcription factors (TFs) → TF genes → target genes. Turning such a
picture into a Bayesian network requires **modelling choices**:

| Choice | Options | Consequence |
|---|---|---|
| variables | expression of genes, phosphorylation of proteins, genotypes | what a node "is" |
| values | discrete (off/on, low/medium/high) → CPTs; continuous → e.g. linear-Gaussian CPDs | number of parameters; information lost by discretising |
| edges | from prior knowledge (databases, ChIP-seq) or learned from data (Module 17) | the independence assumptions (Module 16) |

A common continuous CPD is **linear-Gaussian**: a child is a weighted sum of its parents plus noise,

$$X_i = w_{i0} + \sum_{p\in\mathrm{Pa}(X_i)} w_{ip}\,X_p + \varepsilon_i, \qquad \varepsilon_i\sim\mathcal N(0,\sigma_i^2).$$

*In words:* each protein's level is a baseline, plus a contribution from each of its regulators, plus independent
random noise. It needs $|\mathrm{Pa}(X_i)|+2$ numbers per node (weights, intercept, noise variance). The toolkit's
`simulate_sachs_like` generates exactly such data on the Sachs consensus network — a stand-in we can compare with the
real measurements later.

In [ ]:
# ── 7.2 SIMULATED linear-Gaussian data on the consensus network: correlation ≠ edge ──
sim = simulate_sachs_like(np.random.default_rng(SEED + 7), n=2000)
R = np.abs(sim.corr().to_numpy())
idx = {p: i for i, p in enumerate(SACHS_PROTEINS)}
adjacent = np.zeros_like(R, bool)
for a, b in SACHS_CONSENSUS_EDGES:
    adjacent[idx[a], idx[b]] = adjacent[idx[b], idx[a]] = True
upper = np.triu(np.ones_like(R, bool), 1)
r_edge, r_none = R[upper & adjacent], R[upper & ~adjacent]
print(f"linear-Gaussian parameters: {sum(len(sachs_parents[p]) + 2 for p in SACHS_PROTEINS)} "
      f"(a full 11-D Gaussian needs {11 + 11 * 12 // 2})")
print(f"20 pairs joined by an edge : |r| from {r_edge.min():.2f} to {r_edge.max():.2f}")
print(f"35 pairs without an edge   : |r| up to {r_none.max():.2f}; {np.sum(r_none > 0.2)} of them have |r| > 0.2")
print(f"edges weaker than the strongest non-edge: {np.sum(r_edge < r_none.max())} of 20")
print(f"p38–jnk (no edge; both are children of pka and pkc): |r| = {R[idx['p38'], idx['jnk']]:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1.2, 1]})
im = a1.imshow(R, cmap="viridis", vmin=0, vmax=1)
ii, jj = np.nonzero(adjacent)
a1.scatter(jj, ii, marker="s", s=55, facecolors="none", edgecolors="white", lw=1.2)
a1.set_xticks(range(11)); a1.set_xticklabels(SACHS_PROTEINS, rotation=90, fontsize=7)
a1.set_yticks(range(11)); a1.set_yticklabels(SACHS_PROTEINS, fontsize=7)
a1.set_title("|correlation|, SIMULATED data\n(white squares = edges of the generating graph)")
fig.colorbar(im, ax=a1, shrink=0.8)
jitter = np.random.default_rng(SEED)
for k, (vals, col) in enumerate([(r_edge, "tab:blue"), (r_none, "tab:orange")]):
    a2.scatter(vals, k + jitter.uniform(-0.15, 0.15, len(vals)), color=col, s=18)
a2.set_yticks([0, 1]); a2.set_yticklabels(["20 pairs with\nan edge", "35 pairs with\nno edge"])
a2.set_xlabel("|correlation| between the two proteins"); a2.set_ylim(-0.6, 1.6)
a2.set_title("no correlation threshold separates edges from non-edges")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The simulated proteins are generated *only* along the 20 edges of the consensus graph,
yet 13 of the 35 pairs *without* an edge have $|r|>0.2$ — for example p38 and jnk ($|r|=0.30$), which share the two
parents pka and pkc. Meanwhile some true edges are weak (the weakest has $|r|=0.10$), and 7 of the 20 edges are
weaker than the strongest non-edge. Correlation spreads along every path of the network, so no threshold on
correlation can recover the graph. That needs *conditional* independence (Module 16) and structure learning
(Module 17).
The linear-Gaussian network needs 42 parameters, against 77 for an unrestricted 11-dimensional Gaussian (11 means + 66
variances/covariances) — Module 18 starts from that full Gaussian and makes it sparse in a different way.

⚠️ **What an edge means — and what it does not.**
- In a Bayesian network an arrow $X\to Y$ means: *in this probability model*, $Y$'s distribution is specified
  conditionally on $X$ (together with $Y$'s other parents). The **missing** arrows carry the real content: they are
  conditional independence claims.
- An arrow is **not** proof of a physical mechanism (binding, phosphorylation). The same joint distribution can often be
  written with several different arrow patterns (Module 17: Markov equivalence), unmeasured molecules can create
  dependencies between measured ones, and feedback loops cannot be drawn at all.
- A network *learned from data* is a set of hypotheses about dependence. Sachs *et al.* used targeted
  interventions to orient their arrows; Module 17 shows what purely observational data can and cannot tell you.

---
## 📌 Key takeaways

- A **probabilistic graphical model** = probability distribution + graph. Directed acyclic graphs give **Bayesian
  networks**; undirected graphs give Markov random fields (Module 18).
- A **DAG** has no directed cycles; a topological order exists exactly when the graph is acyclic (Kahn's algorithm).
  Know *parents*, *children*, *descendants*.
- Each node carries a **CPD** $P(X_i\mid\mathrm{Pa}(X_i))$; the joint distribution is their product,
  $P(X_1\dots X_n)=\prod_i P(X_i\mid\mathrm{Pa}(X_i))$, e.g. $P(B,\neg E,A,J,\neg M)=0.005355$.
- The **chain rule** is the complete DAG: $2^n-1$ parameters (31 for five binary variables). Missing edges — domain
  knowledge — cut this to 10 for the alarm network and to 230 (from 177,146) for the Sachs signalling network.
- **Constructing a BN**: identify the variables, determine the graph, determine the CPDs.
- A BN is also a **generative recipe**: ancestral sampling in topological order reproduces the joint distribution.
- Naïve Bayes, pedigrees and regulatory networks are Bayesian networks. An edge encodes statistical dependence in a
  model, **not** proof of a biochemical mechanism.

---
## ✅ Checkpoint

1. Write the joint distribution defined by the DAG with edges A → C, B → C, D → C, A → D (slide 3). How many free parameters does it need if all four variables are binary?
   <details><summary>Answer</summary>$P(A)P(B)P(D\mid A)P(C\mid A,B,D)$; parameters $1+1+2+8=12$ (versus $2^4-1=15$ for the full table).</details>
2. Compute $P(\neg B, E, A, \neg J, M)$ in the alarm network by hand.
   <details><summary>Answer</summary>$P(\neg B)P(E)P(A\mid\neg B,E)P(\neg J\mid A)P(M\mid A)=0.95\times0.1\times0.5\times0.3\times0.8=0.0114$.</details>
3. Why does the chain-rule factorization of five binary variables need 31 parameters whatever order you choose for the variables?
   <details><summary>Answer</summary>Every order gives a complete DAG, which can represent any distribution over the 5 variables; such a distribution is a 32-cell table summing to 1, i.e. $2^5-1=31$ free numbers. Only deleting edges saves parameters.</details>
4. A gene has 3 expression levels and 2 regulators with 3 levels each. How many free parameters does its CPT have? And with 4 regulators?
   <details><summary>Answer</summary>$(3-1)\times3^2=18$; with 4 regulators $(3-1)\times3^4=162$ — the table grows exponentially in the number of parents.</details>
5. Kahn's algorithm stops with leftover nodes {E, A, J, M, P} in the alarm network with an extra edge M → E. Are all five on a cycle?
   <details><summary>Answer</summary>No. Only E, A and M lie on the cycle E → A → M → E; J and P are left over because they are downstream of the cycle (their parent A is never released).</details>
6. In the movie network, compute $P(Le{=}1\mid S{=}1)$ and explain in one sentence why $P(Li{=}1\mid S{=}1,Le{=}1)=P(Li{=}1\mid Le{=}1)$.
   <details><summary>Answer</summary>$P(Le{=}1\mid S{=}1)=\frac{0.5\cdot0.6}{0.5\cdot0.6+0.5\cdot0.2}=0.75$. Le is the only route between S and Li (common cause); once Le is known, S carries no further information about Li.</details>
7. Code reading: in `DiscreteBN.sample`, why must the loop run over `self.order` rather than over `self.parents`?
   <details><summary>Answer</summary>To draw a node we need its parents' sampled values already in `out`; the topological order guarantees that parents are drawn before their children (for the alarm dict the two orders happen to coincide, but not in general).</details>
8. Two genes are strongly correlated across 1,000 cells. Does a Bayesian network for these genes need an edge between them?
   <details><summary>Answer</summary>Not necessarily: correlation can come from a shared regulator (common cause) or a longer path; an edge is needed only if no set of other variables makes them conditionally independent (Module 16). Even then, the edge is a statistical dependence, not proof of direct regulation.</details>

## 🧠 Questions & discussion

1. The alarm network assumes that John's call depends only on the alarm. Name a realistic mechanism that would violate this assumption, and say which arrow you would add and how many parameters it costs.
2. Biological signalling is full of feedback loops (e.g. Erk inhibiting Raf). What exactly goes wrong if you put such a loop into a Bayesian network, and how do dynamic Bayesian networks (Modules 19–20) get around it?
3. In the pedigree network the CPT is fixed by Mendel's laws, while in the signalling network it must be learned. Which other biological networks have mechanistically known CPTs, and which must be learned?
4. Discretising protein levels into low/medium/high makes CPTs possible. What information is lost, and how might the choice of thresholds change which edges look important?
5. A database lists 2,000 transcription factor–target interactions from ChIP-seq. Would you use them as the edges of a Bayesian network for an expression data set? What could go wrong?

## 🚀 Bonus — going further

- **Variable order matters for sparsity.** Build the chain-rule factorization of the alarm distribution in the order M, J, A, B, E and, for each factor, drop every conditioning variable that does not change the conditional probability (use `conditional_from_table`). How many parameters does the resulting network need compared with 10?
- **Noisy-OR.** Replace the four-row CPT of A by a noisy-OR model with two numbers (the probability that a burglary alone, or an earthquake alone, fails to trigger the alarm). Which slide values can it reproduce exactly?
- **A three-generation pedigree.** Extend Section 7.1 to grandparents → parents → child (7 nodes) and compute the probability that a grandparent was TT given that the grandchild is TT.
- **Related notebooks in this repository.** [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/) (naïve Bayes as a generative model on real breast-cancer data) and [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/) (probability and likelihood refresher).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 15 — Bayesian networks I: representation           date:
Input:            alarm-network CPTs (slides), movie-example CPTs (slides), Sachs 2005 consensus graph,
                  Mendelian transmission probabilities
Output:           joint probabilities (0.005355, 0.12), 32-row joint table, parameter counts (31 vs 10;
                  177,146 vs 230), 100,000 ancestral samples
Method:           DAG + CPDs; product of local conditionals; Kahn's topological sort; ancestral sampling
Key equation:     P(X1..Xn) = Π_i P(Xi | Pa(Xi));  #params(Xi) = (r_i − 1) Π_{p in Pa} r_p
Key parameters:   CPT entries; number of parents per node; number of values per variable
What changed when I modified P(B) / the number of samples / the number of levels:
One failure mode: a missing edge that is not a true conditional independence (wrong model, however much data)
Connection to 02-620 slides: 008_BN_mz_021825 slides 1–37 (+173)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `008_BN_mz_021825` (course design S. Kim; instructors M. Xu and M. Zhang), slides 1–37 and 173.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 10 (directed graphical models).
- Koller D., Friedman N. (2009) *Probabilistic Graphical Models: Principles and Techniques*. MIT Press — Chapter 3.
- Russell S., Norvig P. (2010) *Artificial Intelligence: A Modern Approach*, 3rd ed. Prentice Hall — Chapter 14 (the burglary/earthquake alarm network).
- Pearl J. (1988) *Probabilistic Reasoning in Intelligent Systems*. Morgan Kaufmann.
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from multiparameter single-cell data. *Science* 308:523–529.
- Kahn A. B. (1962) Topological sorting of large networks. *Communications of the ACM* 5:558–562.

## ➡️ Next

You can now write down a Bayesian network and its joint distribution. But what does the graph *say*? In
[Module 16 · Bayesian networks II: reading independence from the graph](16_bayesian_networks_independence.ipynb)
you will learn to read conditional independence statements directly off the arrows — Markov blankets and
d-separation, with all 15 worked examples of the slides checked by brute force.